# Demo 4 · agentgateway — MCP tools federated across two clusters

**The story for the customer:** an AI agent wants a set of tools, but those tools live in different clusters. agentgateway federates them behind **one** MCP endpoint, reaches the remote ones over HBONE, and governs them all with a **single** policy. The agent never knows there were two clusters.

We run two MCP tool servers, `catalog-mcp` on **mesh1** and `orders-mcp` on **mesh2** (published as a cross-cluster global service). One agentgateway `AgentgatewayBackend` federates both into a single `/mcp` endpoint, then one `AgentgatewayPolicy` allow-lists which tools survive, on both clusters at once.

> **Kernel:** Bash (Select Kernel → Jupyter Kernel → **Bash**).
> This notebook is **self-contained**: run **Connect**, then **Reset**, then the steps top to bottom.
> It needs `./setup.sh` to have been run once. After a laptop sleep: `./demo-scripts/wake.sh`.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 290" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"> <rect x="0" y="0" width="720" height="290" rx="10" fill="#f8fafc"/> <text x="360" y="27" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">One MCP endpoint, tools federated from two clusters</text> <defs> <marker id="fa" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker> </defs> <!-- MCP client --> <rect x="14" y="104" width="86" height="50" rx="8" fill="#e2e8f0" stroke="#64748b"/> <text x="57" y="125" text-anchor="middle" font-size="11" fill="#334155">MCP client</text> <text x="57" y="140" text-anchor="middle" font-size="9.5" fill="#475569">(agent / curl)</text> <line x1="100" y1="129" x2="128" y2="129" stroke="#334155" stroke-width="1.6" marker-end="url(#fa)"/> <!-- mesh1 --> <rect x="128" y="52" width="302" height="192" rx="10" fill="#eff6ff" stroke="#3b82f6" stroke-width="2"/> <text x="279" y="72" text-anchor="middle" font-size="13" font-weight="700" fill="#1e3a8a">mesh1</text> <rect x="146" y="86" width="266" height="52" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.8"/> <text x="279" y="106" text-anchor="middle" font-size="12" font-weight="700" fill="#14532d">agentgateway  ·  one /mcp endpoint</text> <text x="279" y="123" text-anchor="middle" font-size="9.5" fill="#166534">AgentgatewayBackend, sessionRouting: Stateless</text> <line x1="230" y1="138" x2="230" y2="172" stroke="#334155" stroke-width="1.6" marker-end="url(#fa)"/> <text x="196" y="160" text-anchor="middle" font-size="9" fill="#475569">local</text> <rect x="152" y="172" width="156" height="46" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.5"/> <text x="230" y="192" text-anchor="middle" font-size="11.5" font-weight="600" fill="#1e293b">catalog-mcp</text> <text x="230" y="207" text-anchor="middle" font-size="9.5" fill="#475569">tools: catalog_*</text> <!-- HBONE hop in the gap, wall to wall --> <text x="463" y="105" text-anchor="middle" font-size="10.5" font-weight="600" fill="#0f172a">HBONE</text> <line x1="430" y1="115" x2="496" y2="115" stroke="#334155" stroke-width="1.8" marker-end="url(#fa)"/> <text x="463" y="131" text-anchor="middle" font-size="9" fill="#475569">global VIP</text> <!-- mesh2 --> <rect x="496" y="52" width="210" height="192" rx="10" fill="#eff6ff" stroke="#3b82f6" stroke-width="2"/> <text x="601" y="72" text-anchor="middle" font-size="13" font-weight="700" fill="#1e3a8a">mesh2</text> <rect x="514" y="172" width="174" height="46" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.5"/> <text x="601" y="192" text-anchor="middle" font-size="11.5" font-weight="600" fill="#1e293b">orders-mcp</text> <text x="601" y="207" text-anchor="middle" font-size="9.5" fill="#475569">tools: orders_*</text> <rect x="514" y="98" width="174" height="34" rx="7" fill="#fef3c7" stroke="#d97706"/> <text x="601" y="119" text-anchor="middle" font-size="10" fill="#7c2d12">solo.io/service-scope: global</text> <line x1="601" y1="132" x2="601" y2="168" stroke="#334155" stroke-width="1.6" marker-end="url(#fa)"/> <text x="360" y="266" text-anchor="middle" font-size="11" fill="#64748b">The caller sees one endpoint. tools/list returns catalog_* and orders_*; the orders tools execute on mesh2 over HBONE.</text> </svg></div>

### Connect

Sets the contexts and the Solo `istioctl` build, and confirms the clusters are peered. **Run this first, always.**

In [ ]:
# Connect — contexts, the Solo istioctl build, licences. Run this first, always.
[ -d istio-ambient-demo-kind ] && cd istio-ambient-demo-kind || :
export CLUSTER1=kind-mesh1 CLUSTER2=kind-mesh2
export ISTIOCTL=$HOME/.istioctl/bin/istioctl-1.30.3-solo
export SECRETS_FILE="${SECRETS_FILE:-$HOME/code/solo/secrets/secrets-envs.sh}"
[ -f "$SECRETS_FILE" ] && set -a && . "$SECRETS_FILE" && set +a
echo "mesh1=$CLUSTER1  mesh2=$CLUSTER2  licence: $([ -n "$SOLO_ISTIO_LICENSE_KEY" ] && echo yes || echo NO)"
$ISTIOCTL --context $CLUSTER1 multicluster check 2>&1 | grep -E "Peers Check|Gateway Check" || echo "not peered — run ./setup.sh (or ./demo-scripts/wake.sh after a sleep)"


## Reset + deploy · run this to (re)start the demo

Clears anything a previous run left behind (the MCP gateway, backend and policy), then deploys the two tool servers and an MCP client. Safe on a fresh cluster: the deletes are no-ops and the deploys are idempotent. The first `npx` image pull can take ~90s.

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
# clean any leftover Part-4 objects (safe if absent)
kubectl --context $CLUSTER1 -n ai-tools delete agentgatewaypolicy mcp-tool-allowlist --ignore-not-found 2>/dev/null || true
kubectl --context $CLUSTER1 -n ai-tools delete httproute mcp-route --ignore-not-found 2>/dev/null || true
kubectl --context $CLUSTER1 -n ai-tools delete gateway mcp-gateway --ignore-not-found 2>/dev/null || true
kubectl --context $CLUSTER1 -n ai-tools delete agentgatewaybackend mcp-be --ignore-not-found 2>/dev/null || true

# ai-tools namespace on both clusters (ambient + network label, same as bookinfo)
for CTX in $CLUSTER1 $CLUSTER2; do
  kubectl --context $CTX create namespace ai-tools --dry-run=client -o yaml | kubectl --context $CTX apply -f -
  kubectl --context $CTX label namespace ai-tools \
    istio.io/dataplane-mode=ambient topology.istio.io/network=${CTX#kind-} --overwrite
done

# catalog-mcp on mesh1 (local tool server)
kubectl --context $CLUSTER1 -n ai-tools apply -f - <<'EOF'
apiVersion: v1
kind: ServiceAccount
metadata: { name: catalog-mcp }
---
apiVersion: apps/v1
kind: Deployment
metadata: { name: catalog-mcp, labels: { app: catalog-mcp } }
spec:
  replicas: 1
  selector: { matchLabels: { app: catalog-mcp } }
  template:
    metadata: { labels: { app: catalog-mcp } }
    spec:
      serviceAccountName: catalog-mcp
      containers:
      - name: mcp
        image: node:20-alpine
        command: ["npx","-y","@modelcontextprotocol/server-everything","sse"]
        ports: [{ containerPort: 3001 }]
---
apiVersion: v1
kind: Service
metadata: { name: catalog-mcp, labels: { app: catalog-mcp } }
spec:
  selector: { app: catalog-mcp }
  ports: [{ name: http, port: 3001, targetPort: 3001, appProtocol: http }]
EOF

# orders-mcp on mesh2 — the Service is labelled global, so mesh1 reaches it over HBONE
kubectl --context $CLUSTER2 -n ai-tools apply -f - <<'EOF'
apiVersion: v1
kind: ServiceAccount
metadata: { name: orders-mcp }
---
apiVersion: apps/v1
kind: Deployment
metadata: { name: orders-mcp, labels: { app: orders-mcp } }
spec:
  replicas: 1
  selector: { matchLabels: { app: orders-mcp } }
  template:
    metadata: { labels: { app: orders-mcp } }
    spec:
      serviceAccountName: orders-mcp
      containers:
      - name: mcp
        image: node:20-alpine
        command: ["npx","-y","@modelcontextprotocol/server-everything","sse"]
        ports: [{ containerPort: 3001 }]
---
apiVersion: v1
kind: Service
metadata:
  name: orders-mcp
  labels:
    app: orders-mcp
    solo.io/service-scope: global        # published cross-cluster over HBONE
spec:
  selector: { app: orders-mcp }
  ports: [{ name: http, port: 3001, targetPort: 3001, appProtocol: http }]
EOF

echo "  waiting for the tool servers (first npx pull can take ~90s) …"
kubectl --context $CLUSTER1 -n ai-tools wait --for=condition=Ready pod --all --timeout=300s
kubectl --context $CLUSTER2 -n ai-tools wait --for=condition=Ready pod --all --timeout=300s
echo "  ✓ catalog-mcp (mesh1) + orders-mcp (mesh2) ready"


## 4.1 · Federate the two tool servers behind one endpoint

**What we're doing:** put both tool servers behind a single agentgateway `/mcp` endpoint, so one caller sees both clusters' tools.

**How:** an `AgentgatewayBackend` with two MCP targets, `catalog-mcp` locally and `orders-mcp` at its cross-cluster global VIP (reached over HBONE). `sessionRouting: Stateless` means each MCP call is a single request, no session handshake. A `Gateway` (class `enterprise-agentgateway`) + `HTTPRoute` expose it on a LoadBalancer IP.

**What you'll see:** the gateway programmed on a LoadBalancer IP, ready to serve `/mcp`.

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
# mesh2's orders-mcp is a global service; grab the VIP mesh1 routes to it on
VIP=$($ISTIOCTL --context $CLUSTER1 ztunnel-config service 2>/dev/null | awk '/orders-mcp/{print $3}' | cut -d, -f1)
echo "  orders-mcp (mesh2) reachable from mesh1 at $VIP:3001 over HBONE"

kubectl --context $CLUSTER1 -n ai-tools apply -f - <<EOF
apiVersion: agentgateway.dev/v1alpha1
kind: AgentgatewayBackend
metadata: { name: mcp-be }
spec:
  mcp:
    sessionRouting: Stateless
    targets:
    - { name: catalog, static: { host: catalog-mcp.ai-tools.svc.cluster.local, port: 3001, protocol: SSE } }
    - { name: orders,  static: { host: "$VIP", port: 3001, protocol: SSE } }
EOF

kubectl --context $CLUSTER1 -n ai-tools apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata: { name: mcp-gateway, namespace: ai-tools }
spec:
  gatewayClassName: enterprise-agentgateway
  listeners: [{ name: http, port: 8080, protocol: HTTP, allowedRoutes: { namespaces: { from: Same } } }]
---
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: { name: mcp-route, namespace: ai-tools }
spec:
  parentRefs: [{ name: mcp-gateway }]
  rules:
  - backendRefs: [{ group: agentgateway.dev, kind: AgentgatewayBackend, name: mcp-be }]
EOF

kubectl --context $CLUSTER1 -n ai-tools wait --for=condition=Programmed gateway/mcp-gateway --timeout=90s
for i in $(seq 1 40); do
  kubectl --context $CLUSTER1 -n ai-tools get pod -l gateway.networking.k8s.io/gateway-name=mcp-gateway 2>/dev/null | grep -q . && break
  sleep 3
done
kubectl --context $CLUSTER1 -n ai-tools wait --for=condition=Ready pod -l gateway.networking.k8s.io/gateway-name=mcp-gateway --timeout=120s
GW=$(kubectl --context $CLUSTER1 -n ai-tools get gateway mcp-gateway -o jsonpath='{.status.addresses[0].value}')
echo "  ✓ MCP endpoint ready  ·  http://$GW:8080/mcp"


## 4.2 · One `tools/list`, tools from both clusters

**What we're doing:** ask the single endpoint what tools it has.

**How:** MCP speaks JSON-RPC over HTTP; because the backend is `Stateless`, one `POST` lists the tools (the response is an SSE frame, so we strip the leading `data:`).

**What you'll see:** `catalog_*` tools (from mesh1) **and** `orders_*` tools (from mesh2) in one list. The `orders_*` names come from the server running on the other cluster.

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
GW=$(kubectl --context $CLUSTER1 -n ai-tools get gateway mcp-gateway -o jsonpath='{.status.addresses[0].value}')
echo "  ══ tools/list from the one endpoint (catalog_* = mesh1, orders_* = mesh2) ══"
curl -s -X POST "http://$GW:8080/mcp" \
  -H "Content-Type: application/json" -H "Accept: application/json,text/event-stream" \
  -d '{"jsonrpc":"2.0","id":1,"method":"tools/list"}' \
  | sed 's/^data: //' | grep -oE '"name":"[^"]+"' | sed 's/"name":"//; s/"//' \
  | sort | awk '{printf "    %s\n", $0}'


## 4.3 · Call a tool on each cluster

**What we're doing:** invoke one tool that runs locally on mesh1 and one that runs on mesh2, through the same endpoint.

**How:** a JSON-RPC `tools/call` for each. The `orders_echo` call executes on mesh2 and returns over HBONE; the caller uses the identical endpoint for both.

**What you'll see:** a result from each, with the mesh2 one having crossed clusters transparently.

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
GW=$(kubectl --context $CLUSTER1 -n ai-tools get gateway mcp-gateway -o jsonpath='{.status.addresses[0].value}')
call() { curl -s -X POST "http://$GW:8080/mcp" -H "Content-Type: application/json" -H "Accept: application/json,text/event-stream" -d "$1" | sed 's/^data: //'; }
echo "  ══ call a tool on each cluster, through one endpoint ══"
O=$(call '{"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"orders_echo","arguments":{"message":"hi from mesh1"}}}' | grep -oE '"text":"[^"]*"' | head -1 | sed 's/"text":"//; s/"$//')
C=$(call '{"jsonrpc":"2.0","id":3,"method":"tools/call","params":{"name":"catalog_get-sum","arguments":{"a":40,"b":2}}}' | grep -oE '"text":"[^"]*"' | head -1 | sed 's/"text":"//; s/"$//')
printf "    %-26s %s\n" "orders_echo   (mesh2, HBONE)" "$O"
printf "    %-26s %s\n" "catalog_get-sum (mesh1)"      "$C"


## 4.4 · Govern the federated tools with one policy

**What we're doing:** allow-list the tools an agent may use, once, and have it apply to both clusters.

**How:** an `AgentgatewayPolicy` with `action: Allow` is deny-by-default. Only tools matching the CEL rules survive `tools/list`, and a blocked tool cannot be called, across both `catalog` (mesh1) and `orders` (mesh2) targets.

**What you'll see:** the federated list collapses to just the allowed tools on both clusters, and a blocked tool returns an error when called.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 260" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"> <rect x="0" y="0" width="720" height="260" rx="10" fill="#f8fafc"/> <text x="360" y="27" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">One policy governs the federated tools, across both clusters</text> <defs> <marker id="gg" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker> <marker id="gr" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#dc2626"/></marker> </defs> <!-- tools coming in from both clusters --> <rect x="20" y="52" width="180" height="30" rx="7" fill="#dcfce7" stroke="#16a34a"/> <text x="110" y="72" text-anchor="middle" font-size="11" fill="#14532d">catalog_get-sum  (mesh1)</text> <rect x="20" y="90" width="180" height="30" rx="7" fill="#dcfce7" stroke="#16a34a"/> <text x="110" y="110" text-anchor="middle" font-size="11" fill="#14532d">orders_echo  (mesh2)</text> <rect x="20" y="132" width="180" height="30" rx="7" fill="#fee2e2" stroke="#dc2626"/> <text x="110" y="152" text-anchor="middle" font-size="11" fill="#7f1d1d">catalog_get-env  (mesh1)</text> <rect x="20" y="170" width="180" height="30" rx="7" fill="#fee2e2" stroke="#dc2626"/> <text x="110" y="190" text-anchor="middle" font-size="11" fill="#7f1d1d">orders_get-env  (mesh2)</text> <!-- policy gate --> <rect x="290" y="86" width="150" height="82" rx="10" fill="#e0e7ff" stroke="#6366f1" stroke-width="2"/> <text x="365" y="118" text-anchor="middle" font-size="12" font-weight="700" fill="#312e81">AgentgatewayPolicy</text> <text x="365" y="135" text-anchor="middle" font-size="9.5" fill="#4338ca">action: Allow (deny by default)</text> <text x="365" y="149" text-anchor="middle" font-size="9.5" fill="#4338ca">allow: echo, get-sum</text> <line x1="200" y1="67"  x2="288" y2="112" stroke="#16a34a" stroke-width="2" marker-end="url(#gg)"/> <line x1="200" y1="105" x2="288" y2="122" stroke="#16a34a" stroke-width="2" marker-end="url(#gg)"/> <line x1="200" y1="147" x2="288" y2="134" stroke="#dc2626" stroke-width="2" stroke-dasharray="5 3" marker-end="url(#gr)"/> <line x1="200" y1="185" x2="288" y2="144" stroke="#dc2626" stroke-width="2" stroke-dasharray="5 3" marker-end="url(#gr)"/> <!-- allowed set out --> <line x1="440" y1="120" x2="512" y2="120" stroke="#16a34a" stroke-width="2" marker-end="url(#gg)"/> <rect x="512" y="86" width="188" height="70" rx="9" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.5"/> <text x="606" y="110" text-anchor="middle" font-size="11.5" font-weight="600" fill="#1e293b">tools/list returns only:</text> <text x="606" y="128" text-anchor="middle" font-size="10.5" fill="#166534">catalog_get-sum  ·  orders_echo</text> <text x="606" y="144" text-anchor="middle" font-size="9.5" fill="#475569">blocked tools can't even be called</text> <text x="360" y="238" text-anchor="middle" font-size="11" fill="#64748b">Deny-by-default: only tools matching the CEL rules survive tools/list, on both mesh1 and mesh2, from one policy.</text> </svg></div>

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
kubectl --context $CLUSTER1 -n ai-tools apply -f - <<'EOF'
apiVersion: agentgateway.dev/v1alpha1
kind: AgentgatewayPolicy
metadata: { name: mcp-tool-allowlist, namespace: ai-tools }
spec:
  targetRefs: [{ group: agentgateway.dev, kind: AgentgatewayBackend, name: mcp-be }]
  backend:
    mcp:
      authorization:
        action: Allow            # deny-by-default: only matching tools pass
        policy:
          matchExpressions:
          - 'mcp.tool.name == "echo"'
          - 'mcp.tool.name == "get-sum"'
EOF
sleep 3
GW=$(kubectl --context $CLUSTER1 -n ai-tools get gateway mcp-gateway -o jsonpath='{.status.addresses[0].value}')
call() { curl -s -X POST "http://$GW:8080/mcp" -H "Content-Type: application/json" -H "Accept: application/json,text/event-stream" -d "$1" | sed 's/^data: //'; }

echo "  ══ tools/list after the allow-list (both clusters collapse to the allowed set) ══"
call '{"jsonrpc":"2.0","id":1,"method":"tools/list"}' | grep -oE '"name":"[^"]+"' | sed 's/"name":"//; s/"//' | sort | awk '{printf "    %s\n", $0}'

echo
echo "  ══ a blocked tool cannot be called ══"
R=$(call '{"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"orders_get-env","arguments":{}}}' | grep -oE '"(message|text)":"[^"]*"' | head -1)
printf "    %-22s %s\n" "orders_get-env" "${R:-blocked}"


**That is the agentgateway story end to end:** one MCP endpoint, tools federated from two clusters over HBONE, and a single policy governing them all, deny-by-default, across both clusters. Watch it in the Gloo UI service graph (turn **Idle Nodes off**): the `mcp-gateway` fans out to `catalog-mcp` on mesh1 and, cross-cluster, to `orders-mcp` on mesh2.

## 4.R · Reset Demo 4

**What we're doing:** remove the MCP gateway, backend and policy so the demo can be re-run. The tool servers stay deployed.

**How:** delete the policy, route, gateway and backend.

**What you'll see:** the `ai-tools` namespace clean of the gateway objects.

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
kubectl --context $CLUSTER1 -n ai-tools delete agentgatewaypolicy mcp-tool-allowlist --ignore-not-found
kubectl --context $CLUSTER1 -n ai-tools delete httproute mcp-route --ignore-not-found
kubectl --context $CLUSTER1 -n ai-tools delete gateway mcp-gateway --ignore-not-found
kubectl --context $CLUSTER1 -n ai-tools delete agentgatewaybackend mcp-be --ignore-not-found
echo "  ✓ Demo 4 reset (tool servers left deployed)"
